# Artificial Neural Network

## Overview

This notebook is an end-to-end introduction to using an **artificial neural network (ANN) to predict bank customer churn**: whether a customer will leave the bank. It covers data preparation, network construction, training, prediction, and evaluation using `TensorFlow/Keras` and `scikit-learn`.

## 1. Introduction to neural networks

A neural network is a trainable mathematical function that learns relationships between inputs and outputs from examples. Here, the inputs describe a customer, and the output estimates the probability that the customer leaves.

Unlike a hand-written rule such as 'customers with low balances will leave,' the network learns combinations of signals from historical data. These are predictive associations—not necessarily causal relationships.

### What does a neuron calculate?

A neuron combines its inputs using trainable **weights**, adds a **bias**, and applies an **activation function**:

$$
z = \sum_{j=1}^{d} w_jx_j + b, \qquad a = \phi(z)
$$

- $x_j$: an input feature or an output from the previous layer.
- $w_j$: a learned weight controlling that input's contribution.
- $b$: a learned offset.
- $\phi$: an activation function that determines the neuron's output.

Neurons are organized into layers:

- **Input:** the numerical representation of each customer.
- **Hidden layers:** learned transformations that can capture interactions between features.
- **Output layer:** the final prediction.

In a fully connected, or `Dense`, layer, every neuron receives all outputs from the preceding layer. In matrix form:

$$
\mathbf{h}^{(\ell)} = \phi\left(W^{(\ell)}\mathbf{h}^{(\ell-1)} + \mathbf{b}^{(\ell)}\right)
$$

### Why are activation functions important?

Without nonlinear activations, stacking layers would still produce only an affine transformation. Nonlinearities allow the network to learn more complex decision boundaries.

This notebook uses two activations:

**Rectified Linear Unit (ReLU)**, in the hidden layers:

$$
\operatorname{ReLU}(z)=\max(0,z)
$$

**Sigmoid**, in the output layer:

$$
\sigma(z)=\frac{1}{1+e^{-z}}
$$

The `sigmoid` output lies between 0 and 1 and is interpreted as an estimated churn probability. That interpretation does not guarantee that the probabilities are well calibrated.

### How does the network learn?

Training repeatedly follows this cycle:

```mermaid
flowchart LR
    A[Batch of customer records] --> B[Forward pass: predict churn]
    B --> C[Calculate loss against actual outcomes]
    C --> D[Backpropagation: calculate gradients]
    D --> E[Optimizer updates weights and biases]
    E --> B
```

The notebook minimizes **binary cross-entropy**:

$$
\mathcal{L}=-\frac{1}{N}\sum_{i=1}^{N}\left[y_i\log(p_i)+(1-y_i)\log(1-p_i)\right]
$$

Here, $y_i$ is the true label and $p_i$ is the estimated probability of leaving. Confident but incorrect predictions receive a large penalty.

**Backpropagation** calculates how each parameter contributes to the loss. The **Adam** optimizer uses those gradients to update the parameters. Importantly, the model optimizes the loss—not accuracy directly.

The objective is **generalization**: performing well on unseen customers, rather than merely memorizing training examples.

## 2. What the notebook contains

### Part 1 — Data preprocessing

**Loading and selecting data**

The notebook reads `data/Churn_Modelling.csv` and separates predictors from the target:

```python
X = dataset.iloc[:, 3:-1].values
y = dataset.iloc[:, -1].values
```

This excludes `RowNumber`, `CustomerId`, and `Surname`. The ten original predictors are:

| Feature | Meaning |
|---|---|
| `CreditScore` | Customer credit score |
| `Geography` | Country |
| `Gender` | Recorded gender category |
| `Age` | Customer age |
| `Tenure` | Years with the bank |
| `Balance` | Account balance |
| `NumOfProducts` | Number of bank products |
| `HasCrCard` | Whether the customer has a credit card |
| `IsActiveMember` | Whether the customer is active |
| `EstimatedSalary` | Estimated salary |

The target is `Exited`: `1` means the customer left; `0` means they stayed.

**Encoding categorical values**

- `LabelEncoder` converts `Gender` into `Female = 0` and `Male = 1`.
- `OneHotEncoder`, inside a `ColumnTransformer`, expands `Geography` into three indicators: `France`, `Germany`, and `Spain`.

For example, `France` becomes `[1, 0, 0]`. This avoids imposing an artificial numerical ordering on countries.

Replacing one country column with three indicators increases the input width from **10 to 12 features**.

**Splitting and scaling**

The notebook uses an **80% training / 20% test split**, with `random_state=0` fixing the split.

`StandardScaler` then transforms each feature approximately to zero mean and unit variance:

$$
x'_j=\frac{x_j-\mu_{j,\text{train}}}{s_{j,\text{train}}}
$$

Scaling helps optimization when inputs have very different magnitudes, such as age versus salary. The scaler is correctly fitted only on training data and then reused for test data.

### Part 2 — Building the network

The model is a feed-forward `Sequential` network:

```python
ann = tf.keras.models.Sequential()
ann.add(tf.keras.layers.Dense(units=6, activation='relu'))
ann.add(tf.keras.layers.Dense(units=6, activation='relu'))
ann.add(tf.keras.layers.Dense(units=1, activation='sigmoid'))
```

```mermaid
flowchart LR
    A[10 original features] --> B[Encode and standardize]
    B --> C[12 input values]
    C --> D[Hidden layer: 6 ReLU neurons]
    D --> E[Hidden layer: 6 ReLU neurons]
    E --> F[Output: 1 sigmoid neuron]
    F --> G[Estimated probability of leaving]
```

The input shape is inferred when data first reaches the model; there is no explicit `Input` layer in the code.

With 12 inputs, the network has **127 trainable parameters**, including biases:

$$
(12\times6+6)+(6\times6+6)+(6\times1+1)=127
$$

This is a small multilayer perceptron (MLP), suitable for demonstrating the workflow.

### Part 3 — Training

The notebook compiles and trains the model with:

| Setting | Value |
|---|---|
| Optimizer | Adam |
| Loss | `binary_crossentropy` |
| Reported metric | `accuracy` |
| Batch size | 32 |
| Epochs | 100 |

A **batch** is a group of examples processed for one parameter update. An **epoch** is one pass through the training dataset.

The saved final epoch reports training accuracy of **86.21%** and loss of **0.3336**. No validation set or early stopping is used.

### Part 4 — Predictions and evaluation

First, the notebook predicts an example customer: a 40-year-old man in France, with a credit score of 600, three years of tenure, a balance of USD 60,000, two products, a credit card, active membership, and an estimated salary of USD 50,000.

The features are manually encoded and transformed using the training scaler. The saved prediction is `False`: **the model predicts that this customer stays** at the chosen threshold. The actual probability is not displayed.

For all test customers, probabilities are converted into labels using:

```python
y_pred = (ann.predict(X_test) > 0.5)
```

Thus, a probability greater than 0.5 predicts leaving; otherwise, it predicts staying.

## 3. What the saved results mean

The saved test accuracy is **86.3%**, with this confusion matrix:

| Actual outcome | Predicted stays | Predicted leaves |
|---|---:|---:|
| Stays | 1,514 | 81 |
| Leaves | 193 | 212 |

The model correctly classifies 1,726 of 2,000 test customers:

$$
\text{Accuracy}=\frac{1514+212}{2000}=86.3\%
$$

However, accuracy does not tell the whole story. Calculating additional metrics from the saved matrix:

| Metric | Value | Interpretation |
|---|---:|---|
| Churn precision | 72.4% | Of customers flagged as leaving, this proportion actually left |
| Churn recall | 52.3% | Of customers who actually left, this proportion was detected |
| Churn F1 score | 60.7% | Balance between churn precision and recall |

**The key limitation is that the model misses 193 of the 405 customers who leave—almost half.**

Also, simply predicting that every customer stays would achieve **79.75%** accuracy on this test set. The network improves on that baseline by **6.55 percentage points**, but its business usefulness depends on the costs of missed churn and unnecessary retention offers.

## 4. Strengths and limitations

### What the notebook teaches well

- The complete progression from raw tabular data to evaluated predictions.
- Why categorical encoding and feature scaling are needed.
- How hidden `ReLU` layers and a `sigmoid` output support binary classification.
- How to reuse the training scaler for new observations.
- Why evaluation should include a confusion matrix.

### What would strengthen the experiment

- **Fit all preprocessing on training data only.** The categorical encoders are currently fitted before the split. Although they do not use target labels, they expose the preprocessing to test-set category information.
- **Add a validation set and early stopping.** Training loss alone cannot establish whether additional epochs improve generalization.
- **Improve reproducibility.** `random_state=0` fixes the split, but the notebook does not explicitly seed network initialization and training randomness.
- **Evaluate churn-focused metrics and thresholds.** Report precision, recall, F1, and precision–recall performance; choose a threshold using validation data and business costs, not the test set.
- **Automate inference preprocessing.** Reuse the fitted encoders rather than manually constructing country indicators and feature order.
- **Compare simpler and tree-based baselines.** This notebook demonstrates a neural network; it does not establish that one is the best model for this tabular dataset.

**Bottom line:** this is a clear introductory neural-network classification notebook. Its saved results show useful predictive signal, but the 86.3% accuracy should be read alongside the much lower 52.3% churn recall.

In [43]:
import numpy as np
import pandas as pd
import tensorflow as tf

In [44]:
print("TensorFlow version:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

TensorFlow version: 2.21.0
GPUs: []


## Part 1 - Data Preprocessing

### Importing the dataset

- for the **matrix of features**: first 3 columns will be removed because they are not useful for the prediction (id, surname, row number)
- last column will be the **target (dependent) variable vector**

In [45]:
dataset = pd.read_csv('data/Churn_Modelling.csv')
X = dataset.iloc[:, 3:-1].values
y = dataset.iloc[:, -1].values

In [46]:
display(pd.DataFrame(X).head(10))

,0,1,2,3,4,5,6,7,8,9
0,619,France,Female,42,2,0.0,1,1,1,101348.88
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58
2,502,France,Female,42,8,159660.8,3,1,0,113931.57
3,699,France,Female,39,1,0.0,2,0,0,93826.63
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.1
5,645,Spain,Male,44,8,113755.78,2,1,0,149756.71
6,822,France,Male,50,7,0.0,2,1,1,10062.8
7,376,Germany,Female,29,4,115046.74,4,1,0,119346.88
8,501,France,Male,44,4,142051.07,2,0,1,74940.5
9,684,France,Male,27,2,134603.88,1,1,1,71725.73


In [47]:
print(y)

[1 0 1 ... 1 1 0]


### Encoding categorical data

#### Label Encoding the "Gender" column (column index 2 from the new X matrix)

Gender: two categories → one binary column
In this dataset, Gender contains only Female and Male. We are using LabelEncoder to produce:

| Category | Value |
|---|---|
| Female | 0 |
| Male | 1 |

One column is enough to distinguish two possibilities. The numbers don’t mean that one gender is greater or better—they work as an on/off indicator.
Reversing the mapping would work too; the network would simply learn different weights.

#### Geography: three categories → separate indicators

If we would choose label encoding here:

France  → 0
Germany → 1
Spain   → 2

the numerical representation would suggest an artificial order and spacing: Germany sits halfway between France and Spain. That relationship has no
geographical meaning here.

Instead, OneHotEncoder produces:

| Country | France | Germany | Spain |
|---|---|---|---|
| France | 1 | 0 | 0 |
| Germany | 0 | 1 | 0 |
| Spain | 0 | 0 | 1 |

Now each input neuron can learn a separate weight for each country, without assuming an ordering. A nonlinear network could learn from integer-coded
countries, but one-hot encoding avoids making it overcome that misleading structure.

Could gender also be one-hot encoded? Yes. Two indicator columns would work, but one binary column already captures the distinction.

Small API caveat: scikit-learn’s `LabelEncoder` is intended for target labels (y), not input features (X). The tutorial’s binary transformation works, but `OneHotEncoder(drop='if_binary')` is a more appropriate feature-preprocessing option.

In [48]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
X[:, 2] = le.fit_transform(X[:, 2])

In [49]:
display(pd.DataFrame(X).head(10))

,0,1,2,3,4,5,6,7,8,9
0,619,France,0,42,2,0.0,1,1,1,101348.88
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58
2,502,France,0,42,8,159660.8,3,1,0,113931.57
3,699,France,0,39,1,0.0,2,0,0,93826.63
4,850,Spain,0,43,2,125510.82,1,1,1,79084.1
5,645,Spain,1,44,8,113755.78,2,1,0,149756.71
6,822,France,1,50,7,0.0,2,1,1,10062.8
7,376,Germany,0,29,4,115046.74,4,1,0,119346.88
8,501,France,1,44,4,142051.07,2,0,1,74940.5
9,684,France,1,27,2,134603.88,1,1,1,71725.73


In [50]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

ct = ColumnTransformer(transformers=[('encoder', OneHotEncoder(), [1])], remainder='passthrough')
X = np.array(ct.fit_transform(X))

In [51]:
display(pd.DataFrame(X).head(10))

,0,1,2,3,4,5,6,7,8,9,10,11
0,1.0,0.0,0.0,619,0,42,2,0.0,1,1,1,101348.88
1,0.0,0.0,1.0,608,0,41,1,83807.86,1,0,1,112542.58
2,1.0,0.0,0.0,502,0,42,8,159660.8,3,1,0,113931.57
3,1.0,0.0,0.0,699,0,39,1,0.0,2,0,0,93826.63
4,0.0,0.0,1.0,850,0,43,2,125510.82,1,1,1,79084.1
5,0.0,0.0,1.0,645,1,44,8,113755.78,2,1,0,149756.71
6,1.0,0.0,0.0,822,1,50,7,0.0,2,1,1,10062.8
7,0.0,1.0,0.0,376,0,29,4,115046.74,4,1,0,119346.88
8,1.0,0.0,0.0,501,1,44,4,142051.07,2,0,1,74940.5
9,1.0,0.0,0.0,684,1,27,2,134603.88,1,1,1,71725.73


### Splitting the dataset into the Training set and Test set

In [52]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

### Feature Scaling

A mandatory step for neural networks: the optimizer converges faster when features are on similar scales. The network will learn weights more efficiently, and the training process will be more stable.

All features will be scaled to have zero mean and unit variance (not just some of the features). The scaler is fitted on the training data and then applied to both training and test sets. This prevents information from the test set from leaking into the training process.

In [53]:
from sklearn.preprocessing import StandardScaler

sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

## Part 2 - Building the ANN

### Initializing the ANN

In [54]:
ann = tf.keras.models.Sequential()

### Adding the input layer and the first hidden layer

In [55]:
ann.add(tf.keras.layers.Dense(units=6, activation='relu'))

### Adding the second hidden layer

In [56]:
ann.add(tf.keras.layers.Dense(units=6, activation='relu'))

### Adding the output layer

- we only have a binary response (1 or 0), so we only need **one output neuron**;
- also, for this type of output, we use the **sigmoid activation function** instead of ReLU. The sigmoid function outputs a value between 0 and 1, which can be interpreted as a probability;
- in case we had more than two categories, we would have used the **softmax activation function** instead of sigmoid, and the number of output neurons would have been equal to the number of categories;
- for continuous output (regression), we would have used **no activation function** (or a linear activation function) in the output layer;

In [57]:
ann.add(tf.keras.layers.Dense(units=1, activation='sigmoid'))

## Part 3 - Training the ANN

### Compiling the ANN

- for the loss function, we use **binary_crossentropy** for binary classification, **categorical_crossentropy** for multi-class classification, and **mean_squared_error** for regression.

In [58]:
ann.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

### Training the ANN on the Training set

In [59]:
ann.fit(X_train, y_train, batch_size=32, epochs=100)

# we can observe that the accuracy is increasing and the loss is decreasing with each epoch. The final accuracy is around 86% and the loss is around 0.33. It actually converged to this value after about 50 epochs but we trained for 100 epochs to show the training process. In practice, we would use early stopping to stop training when the validation loss stops improving.

Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 552us/step - accuracy: 0.7918 - loss: 0.5196  
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 543us/step - accuracy: 0.7960 - loss: 0.4668
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 530us/step - accuracy: 0.7960 - loss: 0.4481
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 551us/step - accuracy: 0.7974 - loss: 0.4377
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 546us/step - accuracy: 0.7993 - loss: 0.4311
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 520us/step - accuracy: 0.8011 - loss: 0.4251
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 538us/step - accuracy: 0.8065 - loss: 0.4188
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 529us/step - accuracy: 0.8211 - loss: 0.4110
Epoch 9/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 578us/step - accuracy: 0.8267 - loss: 0.4013
Epoch 10/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 568us/step - accuracy: 0.8309 - loss: 0.3907
Epoch 11/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 525us/step - accuracy: 0.8371 - loss: 0.3805
Epoch 12/100
250/

## Part 4 - Making the predictions and evaluating the model

### Predicting the result of a single observation

Use our ANN model to predict if the customer with the following informations will leave the bank:

Geography: France

Credit Score: 600

Gender: Male

Age: 40 years old

Tenure: 3 years

Balance: \$ 60000

Number of Products: 2

Does this customer have a credit card ? Yes

Is this customer an Active Member: Yes

Estimated Salary: \$ 50000

In [60]:
client = [[1, 0, 0, 600, 1, 40, 3, 60000, 2, 1, 1, 50000]]
client = sc.transform(client)
print(ann.predict(client) > 0.5)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
[[False]]


As we chose a sigmoid activation function in the output layer, the output of the ANN is a probability between 0 and 1. We can choose a threshold to decide whether the customer will leave or not. In this case, we chose 0.5 as the threshold. If the predicted probability is greater than 0.5, we predict that the customer will leave (True), otherwise we predict that the customer will stay (False).

---

Therefore, our ANN model predicts that this customer stays in the bank!

**Important note 1:** Notice that the values of the features were all input in a double pair of square brackets. That's because the "predict" method always expects a 2D array as the format of its inputs. And putting our values into a double pair of square brackets makes the input exactly a 2D array.

**Important note 2:** Notice also that the "France" country was not input as a string in the last column but as "1, 0, 0" in the first three columns. That's because of course the predict method expects the one-hot-encoded values of the state, and as we see in the first row of the matrix of features X, "France" was encoded as "1, 0, 0". And be careful to include these values in the first three columns, because the dummy variables are always created in the first columns.

### Predicting the Test set results

In [61]:
y_pred = ann.predict(X_test)
y_pred = (y_pred > 0.5)
y_pred_column = y_pred.reshape(len(y_pred), 1)
y_test_column = y_test.reshape(len(y_test), 1)
print(np.concatenate((y_pred_column, y_test_column), 1))

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 331us/step
[[0 0]
 [0 1]
 [0 0]
 ...
 [0 0]
 [0 0]
 [0 0]]


### Making the Confusion Matrix

In [62]:
from sklearn.metrics import confusion_matrix, accuracy_score

cm = confusion_matrix(y_test, y_pred)
print(cm)
accuracy_score(y_test, y_pred)

[[1524   71]
 [ 208  197]]


0.8605